# UEFA Champions League - XGBoost Poisson Models
## Training, Hyperparameter Tuning & Model Benchmarking

This notebook builds and optimizes separate models for **Home Goals** and **Away Goals**:
- **Target Variables**:
  - `y_home`: Full Time Home Goals (`FTHG`)
  - `y_away`: Full Time Away Goals (`FTAG`)
- **Features**:
  - Categorical: `HomeTeam`, `AwayTeam`
  - Numerical: `Home_GoalsScored_Last5`, `Home_GoalsConceded_Last5`, `Away_GoalsScored_Last5`, `Away_GoalsConceded_Last5`
- **Objective Function**: `count:poisson` (Poisson Regression for counting goals)
- **Tuning**: Grid / RandomizedSearchCV for hyperparameter optimization
- **Benchmarking**: Comparison against Poisson GLM and Random Forest Regressors
- **Inference Verification**: Demonstrating prediction with live rolling form

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import PoissonRegressor
from sklearn.ensemble import RandomForestRegressor
import joblib
from scipy.stats import poisson

In [2]:
df = pd.read_csv("UCL_df.csv")
print("Dataset Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Dataset Shape: (6921, 12)
Columns: ['Date', 'Season', 'Round', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'Home_GoalsScored_Last5', 'Home_GoalsConceded_Last5', 'Away_GoalsScored_Last5', 'Away_GoalsConceded_Last5']


,Date,Season,Round,HomeTeam,AwayTeam,FTHG,FTAG,FTR,Home_GoalsScored_Last5,Home_GoalsConceded_Last5,Away_GoalsScored_Last5,Away_GoalsConceded_Last5
0,1970-08-18,1970,PrelimF,Levski Sofia,Austria Wien,3,1,H,1.433420,1.63342,1.309586,1.595300
1,1970-09-02,1970,PrelimF,Austria Wien,Levski Sofia,3,0,H,1.309586,1.59530,1.433420,1.633420
2,1970-09-16,1970,Round1,Spartak Moskva,Basel,3,2,H,1.633420,1.63342,1.309586,1.881014
3,1970-09-16,1970,Round1,Feyenoord,UT Arad,1,1,D,1.033420,1.03342,0.881014,1.595300
4,1970-09-16,1970,Round1,KF Tirana,Ajax,2,2,D,1.233420,1.63342,1.763925,0.763925


In [3]:
cols_to_drop = ['Date', 'Season', 'Round', 'FTHG', 'FTAG', 'FTR']
X = df.drop(columns=cols_to_drop, errors="ignore")
y_home = df['FTHG']
y_away = df['FTAG']

print("Features (X):")
print(X.head())

Features (X):
         HomeTeam      AwayTeam  Home_GoalsScored_Last5  \
0    Levski Sofia  Austria Wien                1.433420   
1    Austria Wien  Levski Sofia                1.309586   
2  Spartak Moskva         Basel                1.633420   
3       Feyenoord       UT Arad                1.033420   
4       KF Tirana          Ajax                1.233420   

   Home_GoalsConceded_Last5  Away_GoalsScored_Last5  Away_GoalsConceded_Last5  
0                   1.63342                1.309586                  1.595300  
1                   1.59530                1.433420                  1.633420  
2                   1.63342                1.309586                  1.881014  
3                   1.03342                0.881014                  1.595300  
4                   1.63342                1.763925                  0.763925  


In [4]:
n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_home_train = y_home.iloc[:train_end]
y_away_train = y_away.iloc[:train_end]
y_home_val = y_home.iloc[train_end:val_end]
y_away_val = y_away.iloc[train_end:val_end]
y_home_test = y_home.iloc[val_end:]
y_away_test = y_away.iloc[val_end:]

print(f"Total Matches:      {n}")
print(f"Train Set (70%):   {len(X_train)} matches")
print(f"Validation Set (15%): {len(X_val)} matches")
print(f"Test Set (15%):     {len(X_test)} matches")

Total Matches:      6921
Train Set (70%):   4844 matches
Validation Set (15%): 1038 matches
Test Set (15%):     1039 matches


In [5]:
categorical_cols = ['HomeTeam', 'AwayTeam']
numerical_cols = ['Home_GoalsScored_Last5', 'Home_GoalsConceded_Last5', 
                  'Away_GoalsScored_Last5', 'Away_GoalsConceded_Last5']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols),
        ('num', SimpleImputer(strategy='mean'), numerical_cols)
    ]
)

In [6]:
pipe_home = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(objective='count:poisson', n_estimators=1000, max_depth=6, random_state=42))
])

pipe_away = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(objective='count:poisson', n_estimators=1000, max_depth=6, random_state=42))
])

In [7]:
print("Training Baseline Home Model...")
pipe_home.fit(X_train, y_home_train)

print("Training Baseline Away Model...")
pipe_away.fit(X_train, y_away_train)
print("Training Complete!")

Training Baseline Home Model...


Training Baseline Away Model...


Training Complete!


In [8]:
def evaluate_model(pipe, X_data, y_true, set_name="Validation"):
    preds = pipe.predict(X_data)
    mae = mean_absolute_error(y_true, preds)
    mse = mean_squared_error(y_true, preds)
    rmse = np.sqrt(mse)
    
    print(f"--- {set_name} Results ---")
    print(f"   MAE:  {mae:.4f} (Off by {mae:.2f} goals)")
    print(f"   RMSE: {rmse:.4f}")
    return mae, rmse

print("EVALUATING BASELINE ON VALIDATION SET...")
val_mae_home, val_rmse_home = evaluate_model(pipe_home, X_val, y_home_val, "Home Goals (Val)")
val_mae_away, val_rmse_away = evaluate_model(pipe_away, X_val, y_away_val, "Away Goals (Val)")

print("\n" + "="*40 + "\n")

print("EVALUATING BASELINE ON TEST SET...")
test_mae_home, test_rmse_home = evaluate_model(pipe_home, X_test, y_home_test, "Home Goals (Test)")
test_mae_away, test_rmse_away = evaluate_model(pipe_away, X_test, y_away_test, "Away Goals (Test)")

EVALUATING BASELINE ON VALIDATION SET...
--- Home Goals (Val) Results ---
   MAE:  0.9797 (Off by 0.98 goals)
   RMSE: 1.3389
--- Away Goals (Val) Results ---
   MAE:  0.8381 (Off by 0.84 goals)
   RMSE: 1.1688


EVALUATING BASELINE ON TEST SET...
--- Home Goals (Test) Results ---
   MAE:  1.0928 (Off by 1.09 goals)
   RMSE: 1.4264
--- Away Goals (Test) Results ---
   MAE:  1.0231 (Off by 1.02 goals)
   RMSE: 1.4011


In [9]:
joblib.dump(pipe_home, 'UCL_model_home_goals.pkl')
joblib.dump(pipe_away, 'UCL_model_away_goals.pkl')
print("Baseline models saved successfully!")

Baseline models saved successfully!


### Hyperparameter Tuning via Grid / Randomized Search
Optimizing key hyperparameters:
- `n_estimators`: [100, 200, 300, 500]
- `max_depth`: [3, 4, 5, 6]
- `learning_rate`: [0.01, 0.03, 0.05, 0.1]
- `subsample`: [0.7, 0.8, 0.9, 1.0]
- `colsample_bytree`: [0.7, 0.8, 0.9, 1.0]

In [10]:
param_grid = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__max_depth': [3, 4, 5, 6],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

random_search_home = RandomizedSearchCV(
    pipe_home, 
    param_distributions=param_grid, 
    n_iter=20,           
    cv=3,                
    scoring='neg_mean_absolute_error',
    verbose=1,
    random_state=42,
    n_jobs=-1            
)

random_search_away = RandomizedSearchCV(
    pipe_away, 
    param_distributions=param_grid, 
    n_iter=20,
    cv=3,
    scoring='neg_mean_absolute_error',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("Tuning Home Model...")
random_search_home.fit(X_train, y_home_train)
print(f"Best Home Parameters: {random_search_home.best_params_}")
print(f"Best Home Cross-Val MAE: {-random_search_home.best_score_:.4f}")

print("\nTuning Away Model...")
random_search_away.fit(X_train, y_away_train)
print(f"Best Away Parameters: {random_search_away.best_params_}")
print(f"Best Away Cross-Val MAE: {-random_search_away.best_score_:.4f}")

Tuning Home Model...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


Best Home Parameters: {'model__subsample': 0.9, 'model__n_estimators': 300, 'model__max_depth': 5, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.9}
Best Home Cross-Val MAE: 0.8227

Tuning Away Model...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


Best Away Parameters: {'model__subsample': 0.8, 'model__n_estimators': 300, 'model__max_depth': 5, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.8}
Best Away Cross-Val MAE: 0.7321


In [11]:
best_home_model = random_search_home.best_estimator_
best_away_model = random_search_away.best_estimator_

print("RE-EVALUATING OPTIMIZED MODELS ON TEST SET...")
test_mae_home_opt, test_rmse_home_opt = evaluate_model(best_home_model, X_test, y_home_test, "Home Goals (Optimized)")
test_mae_away_opt, test_rmse_away_opt = evaluate_model(best_away_model, X_test, y_away_test, "Away Goals (Optimized)")

print(f"\nImprovement Home MAE: {test_mae_home - test_mae_home_opt:.4f}")
print(f"Improvement Away MAE: {test_mae_away - test_mae_away_opt:.4f}")

RE-EVALUATING OPTIMIZED MODELS ON TEST SET...
--- Home Goals (Optimized) Results ---
   MAE:  0.9813 (Off by 0.98 goals)
   RMSE: 1.2545
--- Away Goals (Optimized) Results ---
   MAE:  0.8720 (Off by 0.87 goals)
   RMSE: 1.1434

Improvement Home MAE: 0.1115
Improvement Away MAE: 0.1511


### Model Comparison & Benchmarking
Comparing the optimized Poisson XGBoost model against Poisson GLM and Random Forest Regressor.

In [12]:
# 1. Poisson GLM
glm_home = Pipeline([('preprocessor', preprocessor), ('model', PoissonRegressor(max_iter=500))])
glm_away = Pipeline([('preprocessor', preprocessor), ('model', PoissonRegressor(max_iter=500))])
glm_home.fit(X_train, y_home_train)
glm_away.fit(X_train, y_away_train)
glm_h_mae = mean_absolute_error(y_home_test, glm_home.predict(X_test))
glm_a_mae = mean_absolute_error(y_away_test, glm_away.predict(X_test))

# 2. Random Forest Regressor
rf_home = Pipeline([('preprocessor', preprocessor), ('model', RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42))])
rf_away = Pipeline([('preprocessor', preprocessor), ('model', RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42))])
rf_home.fit(X_train, y_home_train)
rf_away.fit(X_train, y_away_train)
rf_h_mae = mean_absolute_error(y_home_test, rf_home.predict(X_test))
rf_a_mae = mean_absolute_error(y_away_test, rf_away.predict(X_test))

benchmark_df = pd.DataFrame({
    'Model': ['Poisson GLM', 'Random Forest', 'XGBoost Optimized'],
    'Home Goals Test MAE': [glm_h_mae, rf_h_mae, test_mae_home_opt],
    'Away Goals Test MAE': [glm_a_mae, rf_a_mae, test_mae_away_opt]
})
print("Model Comparison Summary:")
print(benchmark_df.to_string(index=False))

Model Comparison Summary:
            Model  Home Goals Test MAE  Away Goals Test MAE
      Poisson GLM             1.021513             0.978076
    Random Forest             0.986895             0.866780
XGBoost Optimized             0.981269             0.871951


In [13]:
joblib.dump(best_home_model, 'UCL_optimized_model_home.pkl')
joblib.dump(best_away_model, 'UCL_optimized_model_away.pkl')

# Save complete team list
teams_list = sorted(list(set(df['HomeTeam'].unique()) | set(df['AwayTeam'].unique())))
joblib.dump(teams_list, 'teamlist.pkl')
print(f"Saved {len(teams_list)} teams to teamlist.pkl!")
print("Optimized models and team list saved successfully!")

Saved 481 teams to teamlist.pkl!
Optimized models and team list saved successfully!


## Live Match Inference Demonstration
Demonstrating how future predictions are executed by feeding a team's **live rolling form (last 5 games)** into the model to predict Expected Goals (xG) and Poisson win/draw/loss probabilities.

In [14]:
def predict_ucl_fixture(home_team, away_team, h_scored_last5, h_conceded_last5, a_scored_last5, a_conceded_last5):
    input_data = pd.DataFrame({
        'HomeTeam': [home_team],
        'AwayTeam': [away_team],
        'Home_GoalsScored_Last5': [h_scored_last5],
        'Home_GoalsConceded_Last5': [h_conceded_last5],
        'Away_GoalsScored_Last5': [a_scored_last5],
        'Away_GoalsConceded_Last5': [a_conceded_last5],
    })
    
    xg_home = float(best_home_model.predict(input_data)[0])
    xg_away = float(best_away_model.predict(input_data)[0])
    
    MAX_GOALS = 10
    home_probs = poisson.pmf(np.arange(MAX_GOALS), xg_home)
    away_probs = poisson.pmf(np.arange(MAX_GOALS), xg_away)
    prob_matrix = np.outer(home_probs, away_probs)
    
    home_win_prob = float(np.sum(np.tril(prob_matrix, -1))) * 100
    draw_prob = float(np.sum(np.diag(prob_matrix))) * 100
    away_win_prob = float(np.sum(np.triu(prob_matrix, 1))) * 100
    
    print(f"==================================================")
    print(f"Fixture: {home_team} (Home) vs {away_team} (Away)")
    print(f"--------------------------------------------------")
    print(f"Input Form -> {home_team}: {h_scored_last5:.2f} scored / {h_conceded_last5:.2f} conceded (last 5)")
    print(f"Input Form -> {away_team}: {a_scored_last5:.2f} scored / {a_conceded_last5:.2f} conceded (last 5)")
    print(f"Expected Goals (xG): {home_team} {xg_home:.2f} - {xg_away:.2f} {away_team}")
    print(f"Probabilities: Home Win {home_win_prob:.1f}% | Draw {draw_prob:.1f}% | Away Win {away_win_prob:.1f}%")

predict_ucl_fixture('Real Madrid', 'Manchester City', 2.40, 1.00, 2.20, 0.80)
predict_ucl_fixture('Barcelona', 'Paris SG', 2.60, 1.20, 2.10, 0.90)
predict_ucl_fixture('Bayern Munich', 'Arsenal', 2.80, 1.10, 1.90, 0.70)

Fixture: Real Madrid (Home) vs Manchester City (Away)
--------------------------------------------------
Input Form -> Real Madrid: 2.40 scored / 1.00 conceded (last 5)
Input Form -> Manchester City: 2.20 scored / 0.80 conceded (last 5)
Expected Goals (xG): Real Madrid 1.97 - 0.94 Manchester City
Probabilities: Home Win 61.5% | Draw 21.2% | Away Win 17.2%
Fixture: Barcelona (Home) vs Paris SG (Away)
--------------------------------------------------
Input Form -> Barcelona: 2.60 scored / 1.20 conceded (last 5)
Input Form -> Paris SG: 2.10 scored / 0.90 conceded (last 5)
Expected Goals (xG): Barcelona 1.54 - 1.30 Paris SG
Probabilities: Home Win 42.9% | Draw 24.9% | Away Win 32.2%
Fixture: Bayern Munich (Home) vs Arsenal (Away)
--------------------------------------------------
Input Form -> Bayern Munich: 2.80 scored / 1.10 conceded (last 5)
Input Form -> Arsenal: 1.90 scored / 0.70 conceded (last 5)
Expected Goals (xG): Bayern Munich 0.89 - 1.07 Arsenal
Probabilities: Home Win 29.7% |